# 02 - Data Cleaning & Feature Engineering

Demonstrates ERA5 unit conversion, sentinel handling, temporal features, polar-night detection, and extreme-cold flags.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

## 1. Clean the Raw ERA5 Weather Data

In [ ]:
from src.data.cleaning import clean_era5

era5_path = PROJECT_ROOT / "data" / "raw" / "ERA5" / "ERA5_raw_original.csv"
df_era5_raw = pd.read_csv(era5_path, nrows=2000)
df_weather = clean_era5(df_era5_raw)

print("Raw shape:", df_era5_raw.shape)
print("Cleaned shape:", df_weather.shape)
display(df_weather.head())

## 2. Add Temporal and Antarctic Features

In [ ]:
from src.data.feature_engineering import create_features

df_features = create_features(df_weather)
print("Engineered columns:", sorted(set(df_features.columns) - set(df_weather.columns)))
display(df_features[["timestamp", "hour", "season", "polar_night", "extreme_cold"]].head(10))

## 3. Inspect Polar Night and Extreme-Cold Indicators

In [ ]:
plot_df = df_features.iloc[:720].copy()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(plot_df["timestamp"], plot_df["solar_radiation"], color="orange", label="Solar radiation (W/m²)")
ax1.set_xlabel("Timestamp")
ax1.set_ylabel("Solar radiation (W/m²)")

ax2 = ax1.twinx()
ax2.plot(plot_df["timestamp"], plot_df["polar_night"] * 100, color="navy", linestyle="--", label="Polar-night flag (x100)")
ax2.set_ylabel("Polar-night indicator")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.set_title("Solar Irradiance and Polar-Night Detection")
fig.tight_layout()
plt.show()

print("Extreme-cold rows:", int(df_features["extreme_cold"].sum()))